# Task 3 — Consultas analíticas e dashboard

Consome o star schema da Task 2 via Amazon Athena + dashboard interativo Jupyter.

Portátil local↔SageMaker: profile via env var (`AWS_PROFILE=projetos` local; sem env no SageMaker → usa role).

## 0 — Setup

In [33]:
# %pip install -q awswrangler pandas seaborn matplotlib ipywidgets pyarrow

In [34]:
import os
from pathlib import Path

import awswrangler as wr
import boto3
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from ipywidgets import (
    Button, Checkbox, Dropdown, IntSlider, Label, Layout, Text,
    SelectMultiple, SelectionRangeSlider, VBox, HBox, Output, interactive_output,
)
from IPython.display import display

sns.set_theme(style="whitegrid")


In [35]:
# Lê src/.env (gerado por Terraform) se rodar local. No SageMaker pula.
env_path = Path("../src/.env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            k, _, v = line.partition("=")
            os.environ.setdefault(k.strip(), v.strip())

GLUE_DATABASE          = os.environ.get("GLUE_DATABASE", "classicmodels_analytics")
ATHENA_WORKGROUP       = os.environ.get("ATHENA_WORKGROUP", "classicmodels-lab")
ATHENA_OUTPUT_LOCATION = os.environ.get(
    "ATHENA_OUTPUT_LOCATION",
    "s3://lab-classicmodels-gt-projetos-athena-results/results/",
)

# Profile: força "projetos" local (default config quebrada com region inválida).
# No SageMaker, exportar IN_SAGEMAKER=1 para cair na role automática.
if os.environ.get("IN_SAGEMAKER") == "1":
    AWS_PROFILE = None
else:
    AWS_PROFILE = os.environ.get("AWS_PROFILE", "projetos")

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")

boto3_session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)

print(f"GLUE_DATABASE          = {GLUE_DATABASE}")
print(f"ATHENA_WORKGROUP       = {ATHENA_WORKGROUP}")
print(f"ATHENA_OUTPUT_LOCATION = {ATHENA_OUTPUT_LOCATION}")
print(f"AWS_PROFILE            = {AWS_PROFILE or '(role)'}")
print(f"AWS_REGION             = {AWS_REGION}")

GLUE_DATABASE          = classicmodels_analytics
ATHENA_WORKGROUP       = classicmodels-lab
ATHENA_OUTPUT_LOCATION = s3://lab-classicmodels-gt-projetos-athena-results/results/
AWS_PROFILE            = projetos
AWS_REGION             = us-east-1


In [36]:
def run_query(sql: str) -> pd.DataFrame:
    """Executa SQL no Athena e retorna DataFrame."""
    return wr.athena.read_sql_query(
        sql=sql,
        database=GLUE_DATABASE,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT_LOCATION,
        boto3_session=boto3_session,
    )

## 4.2 — Consulta exploratória em `dim_products`

In [37]:
sql_products = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

df_products = run_query(sql_products)
df_products.head()

,product_id,product_name,product_line,product_vendor
0,S10_1949,1952 Alpine Renault 1300,Classic Cars,Classic Metal Creations
1,S12_1666,1958 Setra Bus,Trucks and Buses,Welly Diecast Productions
2,S12_4473,1957 Chevy Pickup,Trucks and Buses,Exoto Designs
3,S18_1097,1940 Ford Pickup Truck,Trucks and Buses,Studio M Art Models
4,S10_4757,1972 Alfa Romeo GTA,Classic Cars,Motor City Art Classics


## 4.3 — Vendas totais por país

In [38]:
sql_country = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_country = run_query(sql_country)
df_country

,country,total_sales
0,Spain,39578007.24
1,USA,29459520.45
2,Singapore,21383820.18
3,France,9066366.18
4,Germany,7072955.64
5,Australia,5063243.31
6,New Zealand,4291623.09
7,UK,3932526.96
8,Switzerland,3916005.12
9,Italy,3245551.29


## 4.4 — Detalhamento (data, linha, produto, país)

In [39]:
sql_detail = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products  ON fact_orders.product_id     = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key    = dim_countries.country_key
JOIN dim_dates     ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

df_detail = run_query(sql_detail)
df_detail["full_date"]   = pd.to_datetime(df_detail["full_date"])
df_detail["total_sales"] = df_detail["total_sales"].astype(float)
print(f"linhas: {len(df_detail)}")
print(df_detail.dtypes)
df_detail.head()

linhas: 2996
full_date       datetime64[s]
product_line           string
product_name           string
country                string
total_sales           float64
dtype: object


,full_date,product_line,product_name,country,total_sales
0,2005-05-29,Classic Cars,1969 Chevrolet Camaro Z28,Australia,218436.75
1,2004-07-19,Trucks and Buses,1996 Peterbilt 379 Stake Bed with Outrigger,Australia,89064.36
2,2004-07-19,Classic Cars,1982 Camaro Z28,Australia,376884.90
3,2004-12-17,Motorcycles,1969 Harley Davidson Ultimate Chopper,Australia,150384.60
4,2004-12-17,Planes,American Airlines: B767-300,Australia,143532.00


## 4.5 — Dashboard interativo

Filtros: intervalo de datas, país, linha de produto, Top N. Gráfico de barras horizontais com Top N produtos.

In [40]:
from ipywidgets import Button, Checkbox, Layout as WLayout, Text

# de-dup country (bug em dim_countries: distinct (country, territory) gera dups)
df_detail = df_detail.groupby(
    ["full_date", "product_line", "product_name", "country"],
    as_index=False,
)["total_sales"].sum()

dates_sorted  = sorted(df_detail["full_date"].dt.date.unique())
countries     = sorted(df_detail["country"].unique())
product_lines = sorted(df_detail["product_line"].unique())

# ---------- Multi-select dropdown ----------
def make_multiselect(options, label):
    cbs = {o: Checkbox(value=True, description=o, indent=False,
                       layout=WLayout(width="auto", margin="0")) for o in options}
    search   = Text(placeholder="buscar...", layout=WLayout(width="220px"))
    btn_all  = Button(description="Todos",  layout=WLayout(width="70px", height="26px"))
    btn_none = Button(description="Nenhum", layout=WLayout(width="70px", height="26px"))

    summary = Button(
        description=f"{label}: Todos ▾",
        layout=WLayout(width="240px", height="32px"),
    )
    list_box = VBox(
        list(cbs.values()),
        layout=WLayout(max_height="220px", overflow_y="auto",
                       border="1px solid #ddd", padding="4px"),
    )
    panel = VBox(
        [search, HBox([btn_all, btn_none]), list_box],
        layout=WLayout(display="none", border="1px solid #aaa",
                       padding="6px", width="260px"),
    )

    def toggle(_):
        panel.layout.display = "none" if panel.layout.display != "none" else "flex"
    summary.on_click(toggle)

    def refresh():
        n = sum(cb.value for cb in cbs.values())
        total = len(cbs)
        txt = "Todos" if n == total else ("Nenhum" if n == 0 else f"{n}/{total}")
        summary.description = f"{label}: {txt} ▾"

    for cb in cbs.values():
        cb.observe(lambda _: refresh(), names="value")

    btn_all.on_click(lambda _: [setattr(cb, "value", True)  for cb in cbs.values()])
    btn_none.on_click(lambda _: [setattr(cb, "value", False) for cb in cbs.values()])

    def on_search(change):
        q = change["new"].lower()
        for opt, cb in cbs.items():
            cb.layout.display = "" if q in opt.lower() else "none"
    search.observe(on_search, names="value")

    return VBox([summary, panel]), cbs

# ---------- Widgets ----------
date_range = SelectionRangeSlider(
    options=[(d.isoformat(), d) for d in dates_sorted],
    index=(0, len(dates_sorted) - 1),
    description="Datas:",
    continuous_update=False,
    layout=WLayout(width="500px"),
)
top_n = IntSlider(value=10, min=1, max=10, description="Top N:", layout=WLayout(width="320px"))

countries_w, country_cbs = make_multiselect(countries,     "Países")
lines_w,     line_cbs    = make_multiselect(product_lines, "Linhas")

out = Output()

def _redraw(*_):
    sel_c = [c for c, cb in country_cbs.items() if cb.value]
    sel_l = [l for l, cb in line_cbs.items()    if cb.value]
    d0, d1 = pd.to_datetime(date_range.value[0]), pd.to_datetime(date_range.value[1])
    df = df_detail[
        (df_detail["full_date"] >= d0) &
        (df_detail["full_date"] <= d1) &
        (df_detail["country"].isin(sel_c)) &
        (df_detail["product_line"].isin(sel_l))
    ]
    with out:
        out.clear_output(wait=True)
        if df.empty:
            print("Sem dados para filtros selecionados.")
            return
        n = top_n.value
        top = df.groupby("product_name", as_index=False)["total_sales"].sum().nlargest(n, "total_sales")
        fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(top))))
        sns.barplot(data=top, x="total_sales", y="product_name",
                    hue="product_name", legend=False, ax=ax, palette="viridis")
        ax.set_xlabel("Vendas totais (sales_amount)")
        ax.set_ylabel("Produto")
        ax.set_title(f"Top {n} produtos — {d0.date()} a {d1.date()}")
        plt.tight_layout()
        plt.show()

for cb in country_cbs.values(): cb.observe(lambda c: _redraw(), names="value")
for cb in line_cbs.values():    cb.observe(lambda c: _redraw(), names="value")
date_range.observe(lambda c: _redraw(), names="value")
top_n.observe(lambda c: _redraw(), names="value")

_redraw()

display(VBox([
    HBox([date_range, top_n]),
    HBox([countries_w, lines_w]),
    out,
]))
